In [6]:
import pandas as pd
import pandas_ta_classic as ta
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau


import numpy as np

import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, precision_score, recall_score, f1_score, matthews_corrcoef


In [ ]:
import os
from pathlib import Path

PERSIST_ROOT = Path(os.environ.get('PERSIST_ROOT', '/mnt/primary'))
if not PERSIST_ROOT.exists():
    raise RuntimeError(f'Persistent storage not found at {PERSIST_ROOT}. Check mounts (df -h /mnt/primary).')

RUN_ROOT = Path(os.environ.get('RUN_ROOT', PERSIST_ROOT / 'metafeatures'))
if not str(RUN_ROOT).startswith(str(PERSIST_ROOT)):
    print(f'WARNING: RUN_ROOT={RUN_ROOT} is not on persistent storage; forcing to {PERSIST_ROOT}/metafeatures')
    RUN_ROOT = PERSIST_ROOT / 'metafeatures'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path(os.environ.get('DATA_PATH', RUN_ROOT / 'ta_nlp_sector.parquet'))
RESULTS_ROOT = Path(os.environ.get('RESULTS_ROOT', RUN_ROOT / 'results'))
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
META_OUT_PATH = Path(os.environ.get('META_OUT_PATH', RUN_ROOT / 'master_df.parquet'))
META_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('DATA_PATH:', DATA_PATH)
print('RESULTS_ROOT:', RESULTS_ROOT)
print('META_OUT_PATH:', META_OUT_PATH)

# DataLoader tuning for 4 CPU cores (adjust if needed)
NUM_WORKERS_TRAIN = 2
NUM_WORKERS_EVAL = 1
PIN_MEMORY = True
PERSISTENT_WORKERS = True

def _dl_kwargs(num_workers: int):
    return dict(
        num_workers=num_workers,
        pin_memory=PIN_MEMORY,
        persistent_workers=(PERSISTENT_WORKERS and num_workers > 0)
    )


In [7]:
master_df = pd.read_parquet(DATA_PATH)
master_df.reset_index(drop=True, inplace=True)

print("shape before sector features:", master_df.shape)
print("Columns in master_df:", master_df.columns)


shape before sector features: (108592, 79)
Columns in master_df: Index(['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker',
       'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9',
       'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3_3', 'stochrsid_14_14_3_3',
       'atrr_14', 'bb_upper', 'bb_middle', 'bb_lower', 'obv', 'ret_1d',
       'roll_ret_1d', 'roll_ret_5d', 'roll_ret_20d', 'sentiment',
       'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
       'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
       'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
       'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
       'emotion_surprize_pct', 'positive_emotion', 'negative_emotion',
       'uncertainty_emotion', 'positive_emotion_pct', 'negative_emotion_pct',
       'uncertainty_emotion_pct', 'stance_label', 'finbert_label',
       'stance_score', 'finbert_score', 'finbert_up', 'finbert_down',

In [8]:
master_df['date'] = pd.to_datetime(master_df['date'])
master_df = master_df.sort_values(by=['ticker', 'date']).reset_index(drop=True)

In [9]:
H_META = 1 # 1-day ahead meta target

In [10]:
master_df = master_df.sort_values(by=['ticker', 'date'])

master_df['ret_1d_meta'] = (
    master_df.groupby('ticker')['close']
    .pct_change(periods=-H_META)
)

/var/folders/jx/mdk91y8925ncdd32prjkcl_c0000gn/T/ipykernel_12215/1110954974.py:5: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=-H_META)


In [11]:
master_df['up_1d_meta'] = (master_df['ret_1d_meta'] > 0).astype(int)

master_df['has_meta_target'] = master_df['ret_1d_meta'].notna().astype(int)

In [7]:
# Feature sets (aligned with Benchmarking.ipynb)
feature_columns = [
    'open', 'high', 'low', 'close', 'volume',
    'roll_ret_1d', 'roll_ret_5d', 'roll_ret_20d',
    'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9',
    'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3_3', 'stochrsid_14_14_3_3',
    'atrr_14', 'bb_upper', 'bb_middle', 'bb_lower', 'obv',
]

sentiment_columns = ['sentiment']

emotion_columns = [
    'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
    'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
    'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
    'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
    'emotion_surprize_pct',
]

unified_emotion_columns = [
    'positive_emotion', 'negative_emotion', 'uncertainty_emotion',
    'positive_emotion_pct', 'negative_emotion_pct', 'uncertainty_emotion_pct',
]

stance_columns = ['stance_label', 'stance_score']

finbert_columns = ['finbert_label', 'finbert_score', 'finbert_up', 'finbert_down', 'finbert_neutral']

sector_columns = [
    'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_close_mean',
    'sector_volume_mean', 'sector_ret_1d', 'sector_ret_5d', 'sector_ret_20d',
    'sector_range', 'sector_vol_20d',
    'ema_12_sector', 'ema_26_sector', 'ema_50_sector', 'macd_12_26_9_sector',
    'macdh_12_26_9_sector', 'macds_12_26_9_sector', 'rsi_14_sector',
    'sector_bb_upper', 'sector_bb_middle', 'sector_bb_lower',
    'market_close', 'sector_rel_strength', 'sector_dispersion_1d',
]

# Feature sets and groups
feature_sets = {
    'base': feature_columns,
    'sentiment': feature_columns + sentiment_columns,
    'emotion': feature_columns + emotion_columns,
    'unified_emotion': feature_columns + unified_emotion_columns,
    'finbert': feature_columns + finbert_columns,
    'all_nlp': feature_columns + sentiment_columns + emotion_columns + unified_emotion_columns + stance_columns + finbert_columns,
    'sector': feature_columns + sector_columns,
    'sector_sentiment': feature_columns + sector_columns + sentiment_columns,
    'sector_emotion': feature_columns + sector_columns + emotion_columns,
    'sector_unified_emotion': feature_columns + sector_columns + unified_emotion_columns,
    'sector_finbert': feature_columns + sector_columns + finbert_columns,
    'sector_all_nlp': feature_columns + sector_columns + sentiment_columns + emotion_columns + unified_emotion_columns + stance_columns + finbert_columns,
    # typo alias used in Benchmarking
    'sentinment': feature_columns + sentiment_columns,
}

feature_groups = {
    'base': ['base'],
    'nlp': ['sentiment', 'emotion', 'unified_emotion', 'finbert', 'all_nlp'],
    'sector': ['sector', 'sector_sentiment', 'sector_emotion', 'sector_unified_emotion', 'sector_finbert', 'sector_all_nlp'],
}

FEATURE_SET_ALIAS = {
    'sentinment': 'sentiment',
}


In [8]:
tickers = master_df['ticker'].unique()
len(tickers), tickers[:5]


(88, array(['AAPL', 'ABB', 'ABBV', 'AEP', 'AGFS'], dtype=object))

In [9]:
def split_ticker_meta(df_ticker, cutoff_date):
    df_ticker = df_ticker.sort_values('date')
    # remove rows without target
    df_ticker = df_ticker[~df_ticker['ret_1d_meta'].isna()].copy()
    if df_ticker.empty:
        return None, None
    
    train_mask = df_ticker['date'] <= cutoff_date
    test_mask  = df_ticker['date'] > cutoff_date
    
    train_df = df_ticker[train_mask]
    test_df  = df_ticker[test_mask]
    
    # need enough data to train
    if len(train_df) < 200 or len(test_df) < 20:
        return None, None
    
    return train_df, test_df


In [10]:
def set_global_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seeds(42)


# Datasets
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class RNNHead(nn.Module):
    # Shared head:
    #   - RNN stack (LSTM/GRU, uni/bi)
    #   - BatchNorm + Dense(32, ReLU) + Dropout
    #   - Output layer (1 unit): linear (regression) or logits (classification)
    def __init__(self, input_size, rnn_type='LSTM', bidirectional=False, problem_type='classification',
                 n_classes=6, ordinal_head='coral', hidden1=128, hidden2=64, num_layers=1,
                 inter_rnn_drop=0.1, dropout=0.3, use_layernorm=False):
        super().__init__()
        self.problem_type = problem_type
        self.bidirectional = bidirectional
        self.rnn_type = rnn_type.upper()
        self.n_classes = n_classes
        self.ordinal_head = ordinal_head
        self.num_layers = int(num_layers)
        self.hidden1 = int(hidden1)
        self.hidden2 = int(hidden2)

        if self.num_layers not in (1, 2):
            raise ValueError("num_layers must be 1 or 2")

        rnn_cls = {'LSTM': nn.LSTM, 'GRU': nn.GRU}[('GRU' if 'GRU' in self.rnn_type else 'LSTM')]

        self.rnn1 = rnn_cls(
            input_size=input_size, hidden_size=self.hidden1, num_layers=1,
            batch_first=True, dropout=0.0, bidirectional=bidirectional
        )

        self.inter_rnn_drop = nn.Dropout(float(inter_rnn_drop))

        self.rnn2 = None
        if self.num_layers == 2:
            self.rnn2 = rnn_cls(
                input_size=self.hidden1*(2 if bidirectional else 1), hidden_size=self.hidden2, num_layers=1,
                batch_first=True, dropout=0.0, bidirectional=bidirectional
            )
            feat_dim = self.hidden2*(2 if bidirectional else 1)
        else:
            feat_dim = self.hidden1*(2 if bidirectional else 1)

        if use_layernorm:
            self.bn = nn.LayerNorm(feat_dim)
        else:
            self.bn = nn.BatchNorm1d(feat_dim)

        self.fc = nn.Linear(feat_dim, 32)
        self.drop = nn.Dropout(float(dropout))
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        # x: [B, T, F]
        out, _ = self.rnn1(x)
        if self.num_layers == 2:
            out = self.inter_rnn_drop(out)   # inter-layer dropout (sequence-wise)
            out, _ = self.rnn2(out)
        # take last timestep: [B, T, H] -> [B, H]
        out = out[:, -1, :]
        out = self.bn(out)
        out = F.relu(self.fc(out))
        out = self.drop(out)
        out = self.out(out)  # shape [B,1]
        return out  # regression: raw; classification: logits
# Early Stopping (PyTorch)
class EarlyStopper:
    def __init__(self, patience=15, min_delta=0.0, restore_best=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best = restore_best
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        improved = (self.best_loss - val_loss) > self.min_delta
        if improved:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best:
                # Deep copy state dict
                self.best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.restore_best and self.best_state is not None:
            model.load_state_dict(self.best_state)


In [11]:
def build_model(input_shape, model_type='LSTM', problem_type='regression', hidden1=128, hidden2=64, 
                num_layers=2, inter_rnn_drop=0.1, dropout=0.3):
    seq_len, n_features = input_shape
    model_type = model_type.upper()
    kwargs = dict(
        problem_type=problem_type,
        hidden1=hidden1,
        hidden2=hidden2,
        num_layers=num_layers,
        inter_rnn_drop=inter_rnn_drop,
        dropout=dropout,
    )
    if model_type == 'LSTM':
        return RNNHead(n_features, rnn_type='LSTM', bidirectional=False, **kwargs)
    elif model_type == 'BILSTM':
        return RNNHead(n_features, rnn_type='LSTM', bidirectional=True, **kwargs)
    elif model_type == 'GRU':
        return RNNHead(n_features, rnn_type='GRU', bidirectional=False, **kwargs)
    elif model_type == 'BIGRU':
        return RNNHead(n_features, rnn_type='GRU', bidirectional=True, **kwargs)
    else:
        raise ValueError("Model type must be one of: ['LSTM','BiLSTM','GRU','BiGRU']")

In [ ]:
# Select best configs (highest mean MCC) per model for H=1
MODELS = ['LSTM', 'BiLSTM', 'GRU', 'BiGRU']
classification_best = select_best_configs('classification', models=MODELS)

# Regression tuning results might not exist; fallback to classification configs
regression_base_dir = RESULTS_ROOT / 'benchmarking' / 'regression'
if not regression_base_dir.exists():
    regression_base_dir = Path('results/benchmarking/regression')
if regression_base_dir.exists():
    regression_best = select_best_configs('regression', models=MODELS)
else:
    print("[WARN] No regression tuning results found under results/benchmarking/regression. Using classification-best configs for regression meta models.")
    regression_best = classification_best

# Print selected configs (ordered)
print("\n=== SELECTED CONFIGS (H=1, sorted by MCC) ===")
for model in MODELS:
    cfg = classification_best[model]
    params = cfg['params']
    print(f"[CLS] {model:<6} | feature_group={cfg['feature_group']:<6} | feature_set={cfg['feature_set']:<18} | mcc_mean={cfg['mcc_mean']:.4f}")
    print(f"      params: seq={params.get('params_sequence_length')} hidden1={params.get('params_hidden1')} hidden2={params.get('params_hidden2')} layers={params.get('params_num_layers')} drop={params.get('params_dropout')} inter_drop={params.get('params_inter_rnn_drop')} lr={params.get('params_learning_rate')} wd={params.get('params_weight_decay')} bs={params.get('params_batch_size')}")

if regression_best is classification_best:
    print("[REG] Using classification-selected configs (no regression tuning results found).")
else:
    for model in MODELS:
        cfg = regression_best[model]
        params = cfg['params']
        print(f"[REG] {model:<6} | feature_group={cfg['feature_group']:<6} | feature_set={cfg['feature_set']:<18} | mcc_mean={cfg['mcc_mean']:.4f}")
        print(f"      params: seq={params.get('params_sequence_length')} hidden1={params.get('params_hidden1')} hidden2={params.get('params_hidden2')} layers={params.get('params_num_layers')} drop={params.get('params_dropout')} inter_drop={params.get('params_inter_rnn_drop')} lr={params.get('params_learning_rate')} wd={params.get('params_weight_decay')} bs={params.get('params_batch_size')}")

# Run walk-forward regression for each model
print(f"\n=== REGRESSION META: refit interval {REFIT_INTERVAL}, warm-up {W_BASE} ===")

all_reg_preds = []
all_reg_metrics = []
skip_counts_reg = {}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

for model in MODELS:
    cfg = regression_best[model]
    params = cfg['params']
    feature_set_name = _resolve_feature_set_name(str(cfg['feature_set']))
    feature_cols = feature_sets[feature_set_name]
    _validate_feature_cols(feature_cols, master_df.columns)

    seq_len = _get_param(params, 'params_sequence_length', L_REGRESSION, int)
    hidden1 = _get_param(params, 'params_hidden1', 64, int)
    hidden2 = _get_param(params, 'params_hidden2', 32, int)
    num_layers = _get_param(params, 'params_num_layers', 2, int)
    inter_rnn_drop = _get_param(params, 'params_inter_rnn_drop', 0.4, float)
    dropout = _get_param(params, 'params_dropout', 0.0, float)
    learning_rate = _get_param(params, 'params_learning_rate', 1e-5, float)
    weight_decay = _get_param(params, 'params_weight_decay', 4e-4, float)
    batch_size = _get_param(params, 'params_batch_size', 32, int)
    max_epochs = _get_param(params, 'params_max_epochs', 50, int)
    early_patience = _get_param(params, 'params_early_stopping_patience', 15, int)
    early_min_delta = _get_param(params, 'params_early_stopping_min_delta', 0.0, float)
    huber_delta = _get_param(params, 'params_huber_delta', 1.0, float)

    print(f"\n[REG-MODEL] {model} | feature_set={feature_set_name} | seq={seq_len} | h1={hidden1} h2={hidden2} | layers={num_layers} | lr={learning_rate} wd={weight_decay} | bs={batch_size}")

    for i, tkr in enumerate(tickers, 1):
        if i == 1 or i % 50 == 0 or i == len(tickers):
            print(f"  [REG] {model} ticker {i}/{len(tickers)}: {tkr}")
        df_tkr = master_df[master_df['ticker'] == tkr].copy()
        pred_df, metrics_df = walk_forward_regression_predictions_per_ticker(
            df_tkr,
            feature_cols=feature_cols,
            target_col='ret_1d_meta',
            model_type=model,
            seq_len=seq_len,
            w_base=W_BASE,
            refit_interval=REFIT_INTERVAL,
            min_seq=MIN_SEQ,
            hidden1=hidden1,
            hidden2=hidden2,
            num_layers=num_layers,
            inter_rnn_drop=inter_rnn_drop,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            max_epochs=max_epochs,
            early_patience=early_patience,
            early_min_delta=early_min_delta,
            huber_delta=huber_delta,
            device=device
        )
        if pred_df is None or pred_df.empty:
            reason = metrics_df if isinstance(metrics_df, str) else 'no_predictions'
            skip_counts_reg[(model, reason)] = skip_counts_reg.get((model, reason), 0) + 1
            continue

        pred_df = pred_df.rename(columns={'regression_pred_ret_1d': f'reg_pred_ret_1d_{model}'})
        all_reg_preds.append(pred_df)
        if metrics_df is not None and not metrics_df.empty:
            metrics_df = metrics_df.copy()
            metrics_df['model'] = model
            all_reg_metrics.append(metrics_df)

# Aggregate regression metrics
if all_reg_metrics:
    metrics_all = pd.concat(all_reg_metrics, ignore_index=True)
    agg = metrics_all[['mae','mse','dir_acc','f1','precision','recall','mcc']].mean().to_dict()
    total_preds = metrics_all['n_preds'].sum()
    print("\n=== REGRESSION META SUMMARY ===")
    print(f"MAE: {agg['mae']:.6f} | MSE: {agg['mse']:.6f} | Dir Acc: {agg['dir_acc']:.4f} | F1: {agg['f1']:.4f} | Precision: {agg['precision']:.4f} | Recall: {agg['recall']:.4f} | MCC: {agg['mcc']:.4f}")
    print(f"Refits: {len(metrics_all)} | Total preds: {total_preds}")

print("Regression skip summary:", skip_counts_reg)


Using GRU meta sequence length: 18, refit interval: 20, warm-up: 300
Processing AAPL for walk-forward GRU predictions...
[META-REG] model=LSTM seq_len=18 w_base=300 refit=20 hidden1=64 hidden2=32 num_layers=2 inter_drop=0.4 dropout=0.0 lr=1e-5 wd=4e-4


KeyboardInterrupt: 

In [ ]:
# Merge regression predictions and compute error/reliability features

# drop any existing regression meta columns
reg_pred_cols = [c for c in master_df.columns if c.startswith('reg_pred_ret_1d_')]
reg_err_cols = [c for c in master_df.columns if c.startswith('reg_')]
cols_to_drop = sorted(set(reg_pred_cols + reg_err_cols))
if cols_to_drop:
    master_df = master_df.drop(columns=cols_to_drop)

# merge new regression predictions
if all_reg_preds:
    # reduce merges to keep memory manageable
    for pred_df in all_reg_preds:
        master_df = master_df.merge(pred_df, on=['date', 'ticker'], how='left')

print('master_df with regression meta:', master_df.shape)

# compute regression prediction errors (diagnostics) per model
master_df = master_df.sort_values(['ticker', 'date']).reset_index(drop=True)
for model in MODELS:
    pred_col = f'reg_pred_ret_1d_{model}'
    if pred_col not in master_df.columns:
        continue
    mask = master_df[pred_col].notna() & master_df['ret_1d_meta'].notna()
    err_col = f'reg_err_ret_1d_{model}'
    abs_col = f'reg_abs_err_ret_1d_{model}'
    sq_col  = f'reg_sq_err_ret_1d_{model}'
    dir_col = f'reg_dir_correct_1d_{model}'

    master_df.loc[mask, err_col] = master_df.loc[mask, pred_col] - master_df.loc[mask, 'ret_1d_meta']
    master_df.loc[mask, abs_col] = master_df.loc[mask, err_col].abs()
    master_df.loc[mask, sq_col]  = master_df.loc[mask, err_col] ** 2
    master_df.loc[mask, dir_col] = (
        (master_df.loc[mask, pred_col] > 0).astype(int) == master_df.loc[mask, 'up_1d_meta']
    ).astype(int)

    # leak-safe reliability features (shifted/rolling)
    master_df[f'reg_abs_err_lag1_{model}'] = master_df.groupby('ticker')[abs_col].shift(1)
    master_df[f'reg_mae_20_{model}'] = (
        master_df.groupby('ticker')[abs_col]
        .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
    )
    master_df[f'reg_rmse_20_{model}'] = (
        master_df.groupby('ticker')[sq_col]
        .transform(lambda s: np.sqrt(s.shift(1).rolling(20, min_periods=5).mean()))
    )
    master_df[f'reg_dir_acc_20_{model}'] = (
        master_df.groupby('ticker')[dir_col]
        .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
    )

print("Regression reliability features computed. Sample:")
cols_preview = ['date','ticker'] + [c for c in master_df.columns if c.startswith('reg_pred_ret_1d_')][:2]
print(master_df[cols_preview].head())


In [ ]:
print(f"\n=== CLASSIFICATION META: refit interval {REFIT_INTERVAL_CLASSIFICATION}, warm-up {W_BASE_CLASSIFICATION} ===")

all_class_preds = []
all_class_metrics = []
skip_counts_class = {}

for model in MODELS:
    cfg = classification_best[model]
    params = cfg['params']
    feature_set_name = _resolve_feature_set_name(str(cfg['feature_set']))
    feature_cols = feature_sets[feature_set_name]
    _validate_feature_cols(feature_cols, master_df.columns)

    seq_len = _get_param(params, 'params_sequence_length', L_CLASSIFICATION, int)
    hidden1 = _get_param(params, 'params_hidden1', 64, int)
    hidden2 = _get_param(params, 'params_hidden2', 32, int)
    num_layers = _get_param(params, 'params_num_layers', 1, int)
    inter_rnn_drop = _get_param(params, 'params_inter_rnn_drop', 0.0, float)
    dropout = _get_param(params, 'params_dropout', 0.4, float)
    learning_rate = _get_param(params, 'params_learning_rate', 1e-3, float)
    weight_decay = _get_param(params, 'params_weight_decay', 1e-4, float)
    batch_size = _get_param(params, 'params_batch_size', 32, int)
    max_epochs = _get_param(params, 'params_max_epochs', 20, int)
    early_patience = _get_param(params, 'params_early_stopping_patience', 5, int)
    early_min_delta = _get_param(params, 'params_early_stopping_min_delta', 0.0, float)

    print(f"\n[CLS-MODEL] {model} | feature_set={feature_set_name} | seq={seq_len} | h1={hidden1} h2={hidden2} | layers={num_layers} | lr={learning_rate} wd={weight_decay} | bs={batch_size}")

    for i, tkr in enumerate(tickers, 1):
        if i == 1 or i % 50 == 0 or i == len(tickers):
            print(f"  [CLS] {model} ticker {i}/{len(tickers)}: {tkr}")
        df_tkr = master_df[master_df['ticker'] == tkr].copy()
        pred_df, metrics_df = walk_forward_classification_predictions_per_ticker(
            df_tkr,
            feature_cols=feature_cols,
            target_col='up_1d_meta',
            model_type=model,
            seq_len=seq_len,
            w_base=W_BASE_CLASSIFICATION,
            refit_interval=REFIT_INTERVAL_CLASSIFICATION,
            min_seq=MIN_SEQ_CLASSIFICATION,
            hidden1=hidden1,
            hidden2=hidden2,
            num_layers=num_layers,
            inter_rnn_drop=inter_rnn_drop,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            max_epochs=max_epochs,
            early_patience=early_patience,
            early_min_delta=early_min_delta,
            device=device
        )
        if pred_df is None or pred_df.empty:
            reason = metrics_df if isinstance(metrics_df, str) else 'no_predictions'
            skip_counts_class[(model, reason)] = skip_counts_class.get((model, reason), 0) + 1
            continue

        pred_df = pred_df.rename(columns={'prob_up_1d': f'cls_prob_up_1d_{model}'})
        all_class_preds.append(pred_df)
        if metrics_df is not None and not metrics_df.empty:
            metrics_df = metrics_df.copy()
            metrics_df['model'] = model
            all_class_metrics.append(metrics_df)

# Aggregate classification metrics
if all_class_metrics:
    metrics_all = pd.concat(all_class_metrics, ignore_index=True)
    agg = metrics_all[['brier','logloss','acc']].mean().to_dict()
    total_preds = metrics_all['n_preds'].sum()
    print("\n=== CLASSIFICATION META SUMMARY ===")
    print(f"Brier: {agg['brier']:.6f} | Logloss: {agg['logloss']:.6f} | Acc: {agg['acc']:.4f}")
    print(f"Refits: {len(metrics_all)} | Total preds: {total_preds}")

print("Classification skip summary:", skip_counts_class)


In [ ]:
# Merge classification predictions and compute error/reliability features

# drop any existing classification meta columns
cls_pred_cols = [c for c in master_df.columns if c.startswith('cls_prob_up_1d_')]
cls_err_cols = [c for c in master_df.columns if c.startswith('cls_')]
cols_to_drop = sorted(set(cls_pred_cols + cls_err_cols))
if cols_to_drop:
    master_df = master_df.drop(columns=cols_to_drop)

# merge new classification predictions
if all_class_preds:
    for pred_df in all_class_preds:
        master_df = master_df.merge(pred_df, on=['date', 'ticker'], how='left')

print('master_df with Classification meta:', master_df.shape)

# diagnostics and leak-safe reliability features for Classification per model
master_df = master_df.sort_values(['ticker','date']).reset_index(drop=True)
for model in MODELS:
    prob_col = f'cls_prob_up_1d_{model}'
    if prob_col not in master_df.columns:
        continue
    mask_cls = master_df[prob_col].notna() & master_df['up_1d_meta'].notna()
    eps = 1e-8
    brier_col = f'cls_brier_{model}'
    logloss_col = f'cls_logloss_{model}'
    correct_col = f'cls_correct_{model}'

    master_df.loc[mask_cls, brier_col] = (master_df.loc[mask_cls, prob_col] - master_df.loc[mask_cls, 'up_1d_meta']) ** 2
    master_df.loc[mask_cls, logloss_col] = -(
        master_df.loc[mask_cls, 'up_1d_meta'] * np.log(master_df.loc[mask_cls, prob_col] + eps)
        + (1 - master_df.loc[mask_cls, 'up_1d_meta']) * np.log(1 - master_df.loc[mask_cls, prob_col] + eps)
    )
    master_df.loc[mask_cls, correct_col] = (
        (master_df.loc[mask_cls, prob_col] >= 0.5).astype(int) == master_df.loc[mask_cls, 'up_1d_meta']
    ).astype(int)

    master_df[f'cls_brier_20_{model}'] = (
        master_df.groupby('ticker')[brier_col]
        .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
    )
    master_df[f'cls_logloss_20_{model}'] = (
        master_df.groupby('ticker')[logloss_col]
        .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
    )
    master_df[f'cls_acc_20_{model}'] = (
        master_df.groupby('ticker')[correct_col]
        .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
    )

print("Classification reliability features computed. Sample:")
cols_preview = ['date','ticker'] + [c for c in master_df.columns if c.startswith('cls_prob_up_1d_')][:2]
print(master_df[cols_preview].head())

# save combined parquet with Regression and Classification signals
out_path = META_OUT_PATH
master_df.to_parquet(out_path, index=False)
print("Saved:", out_path)


In [ ]:
# Quick visual sanity check for one model (best MCC)

try:
    plot_model = max(regression_best, key=lambda m: regression_best[m]['mcc_mean'])
except Exception:
    plot_model = 'LSTM'

pred_col = f'reg_pred_ret_1d_{plot_model}'
abs_err_col = f'reg_abs_err_ret_1d_{plot_model}'

random.seed(42)
sector_ticker_map = {}
for sector in master_df['sector'].dropna().unique():
    tickers_in_sector = master_df.loc[master_df['sector'] == sector, 'ticker'].dropna().unique()
    if len(tickers_in_sector):
        sector_ticker_map[sector] = random.choice(tickers_in_sector)

for sector, ticker in sector_ticker_map.items():
    stock_data = master_df[master_df['ticker'] == ticker].dropna(subset=[pred_col, 'ret_1d_meta'])
    if stock_data.empty:
        print(f"No valid data for sector {sector}, ticker {ticker}. Skipping.")
        continue
    stock_data = stock_data.sort_values('date')
    plt.figure(figsize=(10, 6))
    plt.plot(stock_data['date'], stock_data['ret_1d_meta'], label='Actual Return', color='blue')
    plt.plot(stock_data['date'], stock_data[pred_col], label=f'Regression Pred Return ({plot_model})', color='orange')
    if abs_err_col in stock_data.columns:
        plt.fill_between(
            stock_data['date'],
            stock_data[pred_col] - stock_data[abs_err_col],
            stock_data[pred_col] + stock_data[abs_err_col],
            color='orange', alpha=0.2, label='|Error| Band'
        )
    plt.title(f"{sector} - {ticker}: Actual vs Predicted Return ({plot_model})")
    plt.xlabel("Date")
    plt.ylabel("1-day return")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
# import pandas as pd
# import numpy as np
# import seaborn as sns
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error

# import matplotlib.pyplot as plt

# # Load the dataset
# file_path = "stocknet-dataset/master_df_meta_base.parquet"
# df = pd.read_parquet(file_path)

# # Display basic information about the dataset
# print("Dataset Info:")
# print(df.info())
# print("\nDataset Description:")
# print(df.describe(include='all'))

# # Check for missing values
# missing_values = df.isnull().sum()
# missing_percentage = (missing_values / len(df)) * 100
# missing_df = pd.DataFrame({'Feature': missing_values.index, 'Missing Values': missing_values.values, 'Percentage': missing_percentage.values})
# print("\nMissing Values:")
# print(missing_df.sort_values(by='Percentage', ascending=False))

# # Visualize missing values
# plt.figure(figsize=(10, 6))
# sns.barplot(x='Percentage', y='Feature', data=missing_df.sort_values(by='Percentage', ascending=False).head(20))
# plt.title("Top 20 Features with Missing Values")
# plt.xlabel("Percentage of Missing Values")
# plt.ylabel("Feature")
# plt.show()

# # Correlation analysis
# numerical_features = df.select_dtypes(include=[np.number]).columns
# correlation_matrix = df[numerical_features].corr()

# # Plot heatmap of correlations
# plt.figure(figsize=(12, 10))
# sns.heatmap(correlation_matrix, cmap='coolwarm', annot=False, fmt=".2f")
# plt.title("Correlation Heatmap")
# plt.show()

# # Identify highly correlated features
# threshold = 0.8
# high_corr_pairs = []
# for i in range(len(correlation_matrix.columns)):
#     for j in range(i):
#         if abs(correlation_matrix.iloc[i, j]) > threshold:
#             high_corr_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

# print("\nHighly Correlated Features (Threshold > 0.8):")
# for pair in high_corr_pairs:
#     print(f"{pair[0]} and {pair[1]}: {pair[2]:.2f}")

# # Visualize distributions of numerical features
# for feature in numerical_features:
#     plt.figure(figsize=(8, 4))
#     sns.histplot(df[feature].dropna(), kde=True, bins=30)
#     plt.title(f"Distribution of {feature}")
#     plt.xlabel(feature)
#     plt.ylabel("Frequency")
#     plt.show()